In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = "/content/drive/MyDrive/Architecture_IA/winequality-red.csv"

In [4]:
df = pd.read_csv(data, sep=';')

In [5]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [6]:
df['quality_label'] = df['quality'].apply(lambda x: 1 if x >= 6 else 0)
df['quality_label'].value_counts()

,count
quality_label,
1,855
0,744


In [7]:
X = df.drop(['quality', 'quality_label'], axis=1)
y = df['quality_label']

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important si classes déséquilibrées
)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)

print("Accuracy :", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy : 0.740625
[[111  38]
 [ 45 126]]
              precision    recall  f1-score   support

           0       0.71      0.74      0.73       149
           1       0.77      0.74      0.75       171

    accuracy                           0.74       320
   macro avg       0.74      0.74      0.74       320
weighted avg       0.74      0.74      0.74       320



In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy RF :", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy RF : 0.809375
[[121  28]
 [ 33 138]]
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       149
           1       0.83      0.81      0.82       171

    accuracy                           0.81       320
   macro avg       0.81      0.81      0.81       320
weighted avg       0.81      0.81      0.81       320



In [14]:
from sklearn.metrics import roc_auc_score

y_proba_rf = rf.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba_rf)

print("ROC AUC :", roc_auc)

ROC AUC : 0.904980572235959


In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)

Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [16]:
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))

Accuracy : 0.803125


In [20]:
colonnes = X.columns
nouvelle_donnee = [
    [11.2, 0.28, 0.56, 1.9, 0.075, 17, 60, 0.9980, 3.16, 0.58, 9.8],
    [7.8, 0.88, 0.00, 2.6, 0.098, 25, 67, 0.9968, 3.20, 0.68, 9.8]
]

nouvelle_donnee_df = pd.DataFrame(nouvelle_donnee, columns=colonnes)

nouvelle_donnee_scaled = scaler.transform(nouvelle_donnee_df)

prediction = rf.predict(nouvelle_donnee_scaled)

print("Classe prédite :", prediction)

Classe prédite : [1 0]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import joblib
import os

# Chemin où tu veux sauvegarder
chemin = '/content/drive/MyDrive/Architecture_IA/'

# Vérifier si le dossier existe, sinon le créer
if not os.path.exists(chemin):
    os.makedirs(chemin)

# Sauvegarde du modèle et du scaler dans ce dossier
joblib.dump(rf, os.path.join(chemin, 'random_forest_wine.pkl'))
joblib.dump(scaler, os.path.join(chemin, 'scaler_wine.pkl'))

print("Modèle et scaler sauvegardés dans :", chemin)

Modèle et scaler sauvegardés dans : /content/drive/MyDrive/Architecture_IA/


In [ ]:
import pandas as pd
import joblib


chemin = '/content/drive/MyDrive/Architecture_IA/'

rf_loaded = joblib.load(chemin + 'random_forest_wine.pkl')
scaler_loaded = joblib.load(chemin + 'scaler_wine.pkl')


nouvelle_donnee = [
    [11.2, 0.28, 0.56, 1.9, 0.075, 17, 60, 0.9980, 3.16, 0.58, 9.8],  # Vin 1
    [7.8, 0.88, 0.00, 2.6, 0.098, 25, 67, 0.9968, 3.20, 0.68, 9.8]    # Vin 2
]

# Transformer en DataFrame pour respecter les colonnes d'origine
colonnes = X.columns  # X doit exister ou être défini au préalable
nouvelle_donnee_df = pd.DataFrame(nouvelle_donnee, columns=colonnes)


nouvelle_donnee_scaled = scaler_loaded.transform(nouvelle_donnee_df)


prediction = rf_loaded.predict(nouvelle_donnee_scaled)


for i, p in enumerate(prediction):
    print(f"Vin {i+1} →", "Bon 🍷" if p==1 else "Mauvais 😐")

Vin 1 → Bon 🍷
Vin 2 → Mauvais 😐


In [ ]:
import joblib

# Charger le modèle
rf_loaded = joblib.load('random_forest_wine.pkl')

# Charger le scaler
scaler_loaded = joblib.load('scaler_wine.pkl')

print("Modèle et scaler chargés ✅")

Modèle et scaler chargés ✅


In [21]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("Accuracy XGB :", accuracy_score(y_test, y_pred_xgb))

Accuracy XGB : 0.803125


In [22]:
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

[[122  27]
 [ 36 135]]
              precision    recall  f1-score   support

           0       0.77      0.82      0.79       149
           1       0.83      0.79      0.81       171

    accuracy                           0.80       320
   macro avg       0.80      0.80      0.80       320
weighted avg       0.80      0.80      0.80       320



In [23]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("Accuracy GB:", accuracy_score(y_test, y_pred_gb))

Accuracy GB: 0.78125


In [24]:
print(confusion_matrix(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

[[121  28]
 [ 42 129]]
              precision    recall  f1-score   support

           0       0.74      0.81      0.78       149
           1       0.82      0.75      0.79       171

    accuracy                           0.78       320
   macro avg       0.78      0.78      0.78       320
weighted avg       0.78      0.78      0.78       320



In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=300, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print("CV mean:", scores.mean())
print("CV std:", scores.std())

CV mean: 0.7248275862068965
CV std: 0.02279583637562372
